# Week 32, Day 5: Axons: propagating the action potential, and why myelin helps

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w032/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A spike runs along a squid axon

`axon(d_cm)` simulates a 6 cm Hodgkin-Huxley axon (120 compartments of 0.5 mm) at 6.3 °C, stimulated at one end, and returns the time (ms) at which each compartment first crosses 0 mV. Write `velocity(t_cross, dx)`, which measures the speed between the compartments 30% and 70% of the way along, in m/s (dx in cm), and apply it to a 476 μm squid giant axon.

In [ ]:
import numpy as np

def rates(V):
    V = V + 1e-7
    an = 0.01 * (V + 55) / (1 - np.exp(-(V + 55) / 10)); bn = 0.125 * np.exp(-(V + 65) / 80)
    am = 0.1 * (V + 40) / (1 - np.exp(-(V + 40) / 10)); bm = 4.0 * np.exp(-(V + 65) / 18)
    ah = 0.07 * np.exp(-(V + 65) / 20); bh = 1 / (1 + np.exp(-(V + 35) / 10))
    return an, bn, am, bm, ah, bh

def solve_tri(lower, diag, upper, rhs):
    n = len(diag); c = np.empty(n); dd = np.empty(n)
    c[0], dd[0] = upper[0] / diag[0], rhs[0] / diag[0]
    for i in range(1, n):
        m = diag[i] - lower[i - 1] * c[i - 1]
        c[i] = upper[i] / m if i < n - 1 else 0.0
        dd[i] = (rhs[i] - lower[i - 1] * dd[i - 1]) / m
    x = np.empty(n); x[-1] = dd[-1]
    for i in range(n - 2, -1, -1):
        x[i] = dd[i] - c[i] * x[i + 1]
    return x

def axon(d_cm, Ri=35.4, dx=0.05, N=120, T=12.0, dt=0.01, node_every=None):
    gc = 1000 * d_cm / (4 * Ri * dx**2)                      # coupling, mS/cm^2
    active = np.ones(N, bool) if node_every is None else (np.arange(N) % node_every == 0)
    cm = np.where(active, 1.0, 1 / 50); gl = np.where(active, 0.3, 0.3 / 50)   # myelin: 50 times less C and leak
    V = np.full(N, -65.0); n = np.full(N, 0.3177); m = np.full(N, 0.0529); h = np.full(N, 0.5961)
    off = np.full(N - 1, -gc * dt); diag = cm + 2 * gc * dt; diag[0] -= gc * dt; diag[-1] -= gc * dt
    t_cross = np.full(N, np.nan)
    for s in range(int(T / dt)):
        an, bn, am, bm, ah, bh = rates(V)
        Iion = np.where(active, 120 * m**3 * h * (V - 50) + 36 * n**4 * (V + 77), 0.0) + gl * (V + 54.387)
        Iinj = np.zeros(N)
        if s * dt < 0.5:
            Iinj[:3] = 500.0
        n += dt * (an * (1 - n) - bn * n); m += dt * (am * (1 - m) - bm * m); h += dt * (ah * (1 - h) - bh * h)
        Vnew = solve_tri(off, diag, off, cm * V + dt * (Iinj - Iion))
        new = (Vnew > 0) & (V <= 0) & np.isnan(t_cross)
        t_cross[new] = s * dt
        V = Vnew
    return t_cross

def velocity(t_cross, dx):
    N = len(t_cross); i1, i2 = int(0.3 * N), int(0.7 * N)
    return ...

v_squid = velocity(axon(0.0476), 0.05)
print(f"squid giant axon (476 um) at 6.3 C: {v_squid:.1f} m/s")

In [ ]:
import numpy as np
def test_velocity():
    "velocity divides the distance between the two recording sites by the difference in crossing times, in m/s"
    t = np.full(120, np.nan); t[36] = 2.0; t[84] = 4.0
    assert abs(velocity(t, 0.05) - (84 - 36) * 0.05 * 10 / 2.0) < 1e-9

def test_squid():
    "the 476 um squid axon conducts at 10 to 14 m/s at 6.3 C"
    assert 10 < v_squid < 14

test_velocity()
test_squid()
print("All checks passed")

### Exercise: Measuring the exponent

Measure the velocity for diameters of 119, 238 and 476 μm and fit a straight line to log velocity against log diameter; its slope is the `exponent`.

In [ ]:
diams = np.array([0.0119, 0.0238, 0.0476])
vels = np.array([velocity(axon(d), 0.05) for d in diams])
exponent = ...
for d, v in zip(diams, vels):
    print(f"d = {d * 1e4:4.0f} um: {v:5.2f} m/s")
print(f"velocity grows as diameter to the power {exponent:.2f}")

In [ ]:
import numpy as np
def test_exponent():
    "the fitted exponent of velocity on diameter is between 0.4 and 0.6"
    assert 0.4 < exponent < 0.6

test_exponent()
print("All checks passed")

### Exercise: A model myelinated axon

Measure the velocity of the 119 μm axon without myelin (`v_bare`) and with nodes every 5 compartments, and of the 238 μm axon with nodes every 10 compartments (internodes twice as long, in proportion to diameter). Store the myelinated velocities in `v_my[(d, k)]`.

In [ ]:
v_bare = velocity(axon(0.0119), 0.05)
v_my = {}
for d, k in ((0.0119, 5), (0.0238, 10)):
    v_my[(d, k)] = ...
    print(f"d = {d * 1e4:.0f} um, a node every {k * 0.5:.1f} mm: {v_my[(d, k)]:.1f} m/s")
print(f"unmyelinated 119 um: {v_bare:.1f} m/s;  scaling diameter and internode by 2 changes speed by {v_my[(0.0238, 10)] / v_my[(0.0119, 5)]:.2f}")

In [ ]:
import numpy as np
def test_faster():
    "myelin speeds conduction at the same diameter"
    assert v_my[(0.0119, 5)] > 1.5 * v_bare

def test_linear():
    "doubling diameter and internode length together roughly doubles the speed (ratio between 1.7 and 2.3)"
    assert 1.7 < v_my[(0.0238, 10)] / v_my[(0.0119, 5)] < 2.3

test_faster()
test_linear()
print("All checks passed")